In [0]:
from bggAPI.BGGApi import BGGPlays
from bggAPI.BGGDataCleaner import BGGPlaysCleaner
import math
import time

plays_api = BGGPlays()


In [0]:
game_id_list = spark.read.table("boardgame.boardgamegeek.src__bgg__thing").select("id").collect()

game_id_list = [x.id for x in game_id_list]
game_id_list

In [0]:
fin_data = []
fin_players = []

for game_id in game_id_list:
    print(f"Working with game: {game_id}")
    data = plays_api.get_plays_by_item(game_id)
    if data is not None:
        plays_cleaner = BGGPlaysCleaner(data)

        max_page = math.ceil(plays_cleaner.get_number_of_plays() / 100)
        for i in range(1, max_page+1):
            print(f"getting page {i}/{max_page}")
            data = plays_api.get_plays_by_item(game_id, page=i)
            plays_cleaner = BGGPlaysCleaner(data)

            clean_data, play_players = plays_cleaner.clean_plays()
            fin_data = fin_data + clean_data
            fin_players = fin_players + play_players
            if i % 2 ==0:
                time.sleep(5)
            

In [0]:
if len(fin_data) > 0:
    sp_df = spark.createDataFrame(fin_data)

    sp_df.write.format("delta").mode("append").saveAsTable("boardgame.boardgamegeek.src__bgg__plays")
else:
    print("empty")

In [0]:
if len(fin_players) > 0:
    sp_play_players = spark.createDataFrame(fin_players)

    sp_play_players.write.format("delta").mode("append").saveAsTable("boardgame.boardgamegeek.src__bgg__play_players")